In [1]:

# Cell 1 - Clone repo and install dependencies
!git clone https://github.com/JormayBusso/pixels-to-macros.git
%cd pixels-to-macros

!git fetch --all
!git reset --hard origin/dev

!pip install -q --upgrade pip
!pip install -q segmentation-models-pytorch timm datasets transformers albumentations opencv-python-headless safetensors tqdm Pillow
!grep -vE '^torch|^torchvision|coremltools' training/requirements.txt > /tmp/kaggle_reqs.txt
!pip install -q -r /tmp/kaggle_reqs.txt

import torch
print('CUDA:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
  print(i, torch.cuda.get_device_name(i))


Cloning into 'pixels-to-macros'...
remote: Enumerating objects: 19595, done.
remote: Counting objects: 100% (1638/1638), done.
remote: Compressing objects: 100% (1614/1614), done.
remote: Total 19595 (delta 52), reused 1579 (delta 21), pack-reused 17957 (from 2)
Receiving objects: 100% (19595/19595), 1.63 GiB | 25.00 MiB/s, done.
Resolving deltas: 100% (1262/1262), done.
/kaggle/working/pixels-to-macros
Fetching origin
Updating files: 100% (15949/15949), done.
HEAD is now at 535b1c0c Force 3D scan pipeline contract
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.9 MB/s eta 0:00:00
CUDA: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [2]:
# Cell 2 - Auto-Detect FoodSeg103 and Set Up
from pathlib import Path

# Use the exact path shown in your sidebar
DATA_DIR = Path('/kaggle/input/datasets/jormay/foodseg103-dataset/FoodSeg103')

# Verify it exists
assert DATA_DIR.exists(), f"Dataset not found at {DATA_DIR}. Check the sidebar!"

OUTPUT_DIR = Path('/kaggle/working/pixels-to-macros-segformer-103')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Success! Data found at: {DATA_DIR}")
print(f"🚀 Ready to start training!")

✅ Success! Data found at: /kaggle/input/datasets/jormay/foodseg103-dataset/FoodSeg103
🚀 Ready to start training!


In [3]:
# Cell 3 - Diagnostic Run for Illegal Memory Access
from pathlib import Path
import os

# 1. Force CUDA to stop asynchronously hiding errors
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# 2. Keep the memory fragmentation fix
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

MODEL       = 'nvidia/segformer-b2-finetuned-ade-512-512'
EPOCHS      = 80
BATCH_SIZE  = 4   
IMG_SIZE    = 512
LR          = 6e-5
NUM_WORKERS = 2
MULTIPLIER  = 3
VAL_EVERY   = 3

CHECKPOINT = "/kaggle/input/datasets/jormay/my-food-weights/last_checkpoint (1).pth"

!python training/train.py \
  --data-dir       {DATA_DIR} \
  --output-dir     {OUTPUT_DIR} \
  --model-name     {MODEL} \
  --num-labels     104 \
  --epochs         {EPOCHS} \
  --batch-size     {BATCH_SIZE} \
  --img-size       {IMG_SIZE} \
  --lr             {LR} \
  --workers        {NUM_WORKERS} \
  --val-every      {VAL_EVERY} \
  --virtual-train-multiplier {MULTIPLIER} \
  --checkpoint-seconds 300 \
  --resume {CHECKPOINT} \
  --amp \
  --no-data-parallel

/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `python training/train.py    --data-dir       /kaggle/input/datasets/jormay/foodseg103-dataset/FoodSeg103    --output-dir     /kaggle/working/pixels-to-macros-segformer-103    --model-name     nvidia/segformer-b2-finetuned-ade-512-512    --num-labels     104    --epochs         80    --batch-size     4    --img-size       512    --lr             6e-05    --workers        2    --val-every      3    --virtual-train-multiplier 3    --checkpoint-seconds 300    --resume /kaggle/input/datasets/jormay/my-food-weights/last_checkpoint (1).pth    --amp    --no-data-parallel'


In [4]:
# Cell 4 - Inspect and zip outputs (Make sure this matches OUTPUT_DIR)
!ls -lh /kaggle/working/pixels-to-macros-segformer-103
!tail -n 20 /kaggle/working/pixels-to-macros-segformer-103/metrics.json || true
!cd /kaggle/working && zip -r pixels-to-macros-segformer-103.zip pixels-to-macros-segformer-103

total 0
tail: cannot open '/kaggle/working/pixels-to-macros-segformer-103/metrics.json' for reading: No such file or directory
  adding: pixels-to-macros-segformer-103/ (stored 0%)
